In [ ]:
import pandas as pd
import os
import json
import librosa
import torch
import numpy as np
import ast
import pickle

import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
import os

# Use current repository root by default
desired_path = os.getcwd()
os.chdir(desired_path)
print("Current working directory:", os.getcwd())


# model

In [ ]:
with open('df_input_scaling_bert_w2v2_reduced.pkl', 'rb') as file: # 원하는 임베딩에 맞게 파일 변경하기
    dataset = pickle.load(file)

In [ ]:
torch.manual_seed(52)

In [ ]:
from torch.utils.data import Dataset

class Multimodal_Datasets(Dataset):
    def __init__(self, dataset, split_type, ablate_features=None):
        self.text = dataset.loc['text', split_type].cpu().detach()
        self.audio = dataset.loc['audio', split_type].cpu().detach()
        self.labels = dataset.loc['labels', split_type].cpu().detach()
        self.acoustic_feature = dataset.loc['acoustic_feature', split_type].cpu().detach()
        self.lexical_feature = dataset.loc['lexical_feature', split_type].cpu().detach()
        self.syntactic_feature = dataset.loc['syntactic_feature', split_type].cpu().detach()
        self.n_modalities = 2 # text/ audio

        # Save the original value
        self.original_text = self.text.clone()
        self.original_audio = self.audio.clone()
        self.original_lexical_feature = self.lexical_feature.clone()
        self.original_syntactic_feature = self.syntactic_feature.clone()
        self.original_acoustic_feature = self.acoustic_feature.clone() 

        # Ablate features if specified:
        self.ablate_features = ablate_features
        if ablate_features is not None:
            for feature in ablate_features:
                setattr(self, feature, torch.zeros_like(getattr(self, feature)))

    def get_n_modalities(self):
        return self.n_modalities
    def get_seq_len(self):
        return self.text.shape[1], self.audio.shape[1]
    def get_dim(self):
        return self.text.shape[2], self.audio.shape[2]
    def get_lbl_info(self):
        # return number_of_labels, label_dim
        return self.labels.shape[1], self.labels.shape[2]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        X = (index, self.text[index], self.audio[index], self.acoustic_feature[index],self.lexical_feature[index],self.syntactic_feature[index])
        Y = self.labels[index]
        META = (0, 0, 0)  # 메타 데이터가 없는 경우 기본값 사용
        return X, Y, META
    
    def get_feature_status(self):
        status = {}
        status['text'] = hasattr(self, 'text')
        status['audio'] = hasattr(self, 'audio')
        status['acoustic_feature'] = hasattr(self, 'acoustic_feature')
        status['lexical_feature'] = hasattr(self, 'lexical_feature')
        status['syntactic_feature'] = hasattr(self, 'syntactic_feature')
    
        # Check if any features are ablated
        if self.ablate_features is not None:
            for feature in self.ablate_features:
                if hasattr(self, feature):
                    if torch.all(getattr(self, feature) == 0).item():
                        status[feature] = "ablated"
                    else:
                        status[feature] = "modified"
                    print(f"Original {feature} value:", getattr(self, f"original_{feature}"))
                    print(f"{feature} value after ablation:", getattr(self, feature))
                else:
                    status[feature] = "Not Excluded"
        else:
            status['ablated_features'] = "None"
    
        return status

ablate_features = []

train_data = Multimodal_Datasets(dataset, 'train', ablate_features=[]) # 여기에서 해당 feature 제외
print("train_data info : ")
print(train_data.get_feature_status())

valid_data = Multimodal_Datasets(dataset, 'valid', ablate_features = []) # 여기에서 해당 feature 제외
print("\nvalid_data info : ")
print(valid_data.get_feature_status())

test_data = Multimodal_Datasets(dataset, 'test', ablate_features = []) # 여기에서 해당 feature 제외
print("\ntest_data info : ")
print(test_data.get_feature_status())

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

In [ ]:
import torch
import argparse

parser = argparse.ArgumentParser(description='MulT_ProTACT')
parser.add_argument('-f', default='', type=str)

# Fixed
parser.add_argument('--model', type=str, default='MulT',
                    help='name of the model to use (Transformer, etc.)')

# Tasks
parser.add_argument('--aonly', action='store_true', default='True',
                    help='use the crossmodal fusion into a (default: False)')
parser.add_argument('--lonly', action='store_true', default='True',
                    help='use the crossmodal fusion into l (default: False)')
parser.add_argument('--aligned', action='store_true',
                    help='consider aligned experiment or not (default: False)')
parser.add_argument('--data_path', type=str, default='data',
                    help='path for storing the dataset')

##### Dropouts --> 변경해서 최적값 찾기
parser.add_argument('--attn_dropout', type=float, default=0.3,  # 0.3
                    help='attention dropout (default: 0.1)')
parser.add_argument('--attn_dropout_a', type=float, default=0.3, # 0.3
                    help='attention dropout (for audio) (default: 0.1)')
parser.add_argument('--relu_dropout', type=float, default=0.1, # 0.1
                    help='relu dropout (default: 0.1)')
parser.add_argument('--embed_dropout', type=float, default=0.1, # 0.1
                    help='embedding dropout')
parser.add_argument('--res_dropout', type=float, default=0.3,   # 0.3
                    help='residual block dropout (default: 0.1)')
parser.add_argument('--out_dropout', type=float, default=0.3,   # 0.3
                    help='output layer dropout (default: 0.1)')

###### Architecture --> 변경해서 최적값 찾기
parser.add_argument('--nlevels', type=int, default=5,  # 3 ###커질수록 네트워크의 깊이, 복잡성이 증가 / 학습 속도가 느려짐
                    help='number of layers in the network (default: 5)')
parser.add_argument('--num_heads', type=int, default=2, ### 커질수록 모델이 더 다양한 특징을 학습 / 속도가 느려짐
                    help='number of heads for the transformer network (default: 5)')
parser.add_argument('--attn_mask', action='store_false', ### 사용하지 않으면 모델이 입력 시퀀스의 모든 토큰에 대해 동등한 관심
                    help='use attention mask for Transformer (default: true)')

###### Tuning --> 변경해서 최적값 찾기
parser.add_argument('--batch_size', type=int, default=512, metavar='N',
                    help='batch size (default: 24)')
parser.add_argument('--clip', type=float, default=3.0,
                    help='gradient clip value (default: 0.8)')
parser.add_argument('--lr', type=float, default=5e-5,
                    help='initial learning rate (default: 1e-3)')
parser.add_argument('--optim', type=str, default='Adam', 
                    help='optimizer to use (default: Adam)')
parser.add_argument('--num_epochs', type=int, default=300,
                    help='number of epochs (default: 40)')
parser.add_argument('--when', type=int, default=30,
                    help='when to decay learning rate (default: 20)')
parser.add_argument('--batch_chunk', type=int, default=1,
                    help='number of chunks per batch (default: 1)')

# Logistics
parser.add_argument('--log_interval', type=int, default=150,
                    help='frequency of result logging (default: 30)')
parser.add_argument('--seed', type=int, default=5555,
                    help='random seed')
parser.add_argument('--no_cuda', action='store_true',
                    help='do not use cuda')
parser.add_argument('--name', type=str, default='mult',
                    help='name of the trial (default: "mult")')
args = parser.parse_args()

use_cuda = True

output_dim_dict = {
    'espeak': 5,
}

In [ ]:
import torch.nn.functional as F

def evaluate(model, ctc_a2l_module, criterion, test=False):

    model.eval()
    loader = test_loader if test else valid_loader
    total_loss = 0.0

    results = []
    truths = []

    with torch.no_grad():
        for i_batch, (batch_X, batch_Y, batch_META) in enumerate(loader):
            sample_ind, text, audio, acoustic_feature, lexical_feature, syntactic_feature = batch_X
            eval_attr = batch_Y

            if hyp_params.use_cuda:
                with torch.cuda.device(0):
                    text, audio, acoustic_feature, lexical_feature, syntactic_feature, eval_attr = text.cuda(), audio.cuda(), acoustic_feature.cuda(), lexical_feature.cuda(), syntactic_feature.cuda(), eval_attr.cuda()

            batch_size = text.size(0)
            
            if (ctc_a2l_module is not None):
                ctc_a2l_net = nn.DataParallel(ctc_a2l_module) if batch_size > 10 else ctc_a2l_module
                audio, _ = ctc_a2l_net(audio)     # audio aligned to text

            net = nn.DataParallel(model) if batch_size > 10 else model
            preds, hiddens = net(text, audio, acoustic_feature, lexical_feature, syntactic_feature)
            total_loss += criterion(preds, eval_attr).item() * batch_size

            # Collect the results into dictionary
            results.append(preds)
            truths.append(eval_attr)

    avg_loss = total_loss / (hyp_params.n_test if test else hyp_params.n_valid)

    results = torch.cat(results)
    truths = torch.cat(truths)
    return avg_loss, results, truths

def correlation_coefficient(trait1, trait2): # 특성 간의 선형적인 관계를 측정함. EX) 정확히 비례하거나 반비례해야 함.
    x = trait1
    y = trait2

    # Masking if either x or y is a masked value
    mask_value = -0.
    mask = (x != mask_value) & (y != mask_value)
    
    x_masked = x * mask
    y_masked = y * mask
    
    mx = torch.mean(x_masked)
    my = torch.mean(y_masked)

    xm = x_masked - mx
    ym = y_masked - my

    r_num = torch.sum(xm * ym)
    r_den = torch.sqrt(torch.sum(xm ** 2) * torch.sum(ym ** 2))

    r = torch.tensor(0.)
    r = torch.where(r_den > 0, r_num / r_den, r)
    return r

def cosine_sim(trait1, trait2):
    x = trait1
    y = trait2

    mask_value = 0.
    mask = (x != mask_value) & (y != mask_value)

    x_masked = x * mask
    y_masked = y * mask
    
    normalize_x = F.normalize(x_masked.unsqueeze(0), p=2, dim=1)
    normalize_y = F.normalize(y_masked.unsqueeze(0), p=2, dim=1)

    cos_similarity = torch.sum(normalize_x * normalize_y)
    return cos_similarity

def trait_sim_loss(y_true, y_pred):
    # mask_value = -1
    # mask = y_true != mask_value
    
    # y_trans = y_true[mask].T
    # y_pred_trans = y_pred[mask].T

    y_trans = y_true
    y_pred_trans = y_pred
    
    y_trans = y_trans.t()
    y_pred_trans = y_pred_trans.t()


    sim_loss = 0.0
    cnt = 0.0
    ts_loss = torch.tensor(0.)
    trait_num = 4  # Assuming the trait number

    for i in range(0, trait_num-1):  # start from idx 0, exlucde the last index since we ignore the overall score
        for j in range(i+1, trait_num):
            corr = correlation_coefficient(y_trans[i], y_trans[j])
            sim_loss += torch.where(corr >= 0.90, 1 - cosine_sim(y_pred_trans[i], y_pred_trans[j]), torch.tensor(0.))
            cnt += torch.where(corr >= 0.90, 1, 0)

    ts_loss = torch.where(cnt > 0, sim_loss / cnt, ts_loss)
    return ts_loss

def mse_loss_function(y_true, y_pred):
    mse_loss = F.mse_loss(y_true, y_pred, reduction='mean')
    return mse_loss

def total_loss(y_pred, y_true):
    alpha = 0.7 # default 0.7
    mse_loss = mse_loss_function(y_true, y_pred)
    ts_loss = trait_sim_loss(y_true, y_pred)
    return alpha * mse_loss + (1 - alpha) * ts_loss


In [ ]:
def save_load_name(args, name=''):
    if args.aligned:
        name = name if len(name) > 0 else 'aligned_model'
    elif not args.aligned:
        name = name if len(name) > 0 else 'nonaligned_model'

    return name + '_' + args.model


def save_model(args, model, name=''):
    name = save_load_name(args, name)
    torch.save(model, f'pre_trained_models__real/{name}.pt')


def load_model(args, name=''):
    name = save_load_name(args, name)
    model = torch.load(f'pre_trained_models__real/{name}.pt')
    return model


In [ ]:
###### import torch
from torch import nn
import sys
from src import models
# from src import models_with_trait_attention as models
# from src import ctc
import torch.optim as optim
import numpy as np
import time
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import pickle

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score, f1_score
from src.eval_metrics import *

####################################################################
#
# Hyperparameters
#
####################################################################

hyp_params = args
hyp_params.orig_d_l, hyp_params.orig_d_a = train_data.get_dim()
hyp_params.l_len, hyp_params.a_len = train_data.get_seq_len()
hyp_params.layers = args.nlevels
hyp_params.use_cuda = use_cuda
hyp_params.dataset = dataset
hyp_params.when = args.when
hyp_params.batch_chunk = args.batch_chunk
hyp_params.n_train, hyp_params.n_valid, hyp_params.n_test = len(train_data), len(valid_data), len(test_data)
hyp_params.model = str.upper(args.model.strip())
hyp_params.output_dim = output_dim_dict.get('espeak', 5)


# with torch.autograd.set_detect_anomaly(True):
epoch_loss = 0
model = getattr(models, hyp_params.model+'Model')(hyp_params)
optimizer = getattr(optim, hyp_params.optim)(model.parameters(), lr=hyp_params.lr)

criterion = total_loss

ctc_criterion = None
ctc_a2l_module = None
ctc_a2l_optimizer = None

scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=hyp_params.when, factor=0.1, verbose=True)
settings = {'model': model,
            'optimizer': optimizer,
            'criterion': criterion,
            'ctc_a2l_module': ctc_a2l_module,
            'ctc_a2l_optimizer': ctc_a2l_optimizer,
            'ctc_criterion': ctc_criterion,
            'scheduler': scheduler}

import torch
from torch import nn
import time

best_valid = float('inf') # 초기 검증 손실을 무한대로 설정
early_stopping_counter = 0  # 조기 종료를 위한 카운터 초기화

train_loss_list = []
valid_loss_list = []

for epoch in range(1, hyp_params.num_epochs + 1):
    model.train()
    num_batches = hyp_params.n_train // hyp_params.batch_size
    epoch_loss, epoch_size = 0, 0  # 초기화
    start_time = time.time()

    for i_batch, (batch_X, batch_Y, batch_META) in enumerate(train_loader):
        sample_ind, text, audio, acoustic_feature, lexical_feature, syntactic_feature = batch_X
        eval_attr = batch_Y
        # model.zero_grad()
        if ctc_criterion is not None:
            ctc_a2l_module.zero_grad()

        if hyp_params.use_cuda:
            text, audio, acoustic_feature, lexical_feature, syntactic_feature, eval_attr = text.cuda(), audio.cuda(), acoustic_feature.cuda(), lexical_feature.cuda(), syntactic_feature.cuda(), eval_attr.cuda()

        optimizer.zero_grad()

        batch_size = text.size(0)
        model.cuda()
        net = nn.DataParallel(model) if batch_size > 10 else model
        preds, hiddens = net(text, audio, acoustic_feature, lexical_feature, syntactic_feature)
        raw_loss = criterion(preds, eval_attr)
        combined_loss = raw_loss  # ctc_loss를 포함시키지 않은 버전

        # Loss function backward
        combined_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), hyp_params.clip)
        optimizer.step()

        epoch_loss += combined_loss.item() * batch_size
        epoch_size += batch_size

        if i_batch % hyp_params.log_interval == 0 and i_batch > 0:
            elapsed_time = time.time() - start_time
            print('Epoch {:2d} | Batch {:3d}/{:3d} | Time/Batch(ms) {:5.2f} | Train Loss {:5.4f}'.
                  format(epoch, i_batch, num_batches, elapsed_time * 1000 / hyp_params.log_interval, epoch_loss / epoch_size))
            start_time = time.time()
            

    # 에포크의 모든 배치 처리가 끝난 후 검증 및 테스트 손실을 계산
    val_loss, _, _ = evaluate(model, ctc_a2l_module, criterion, test=False)

    duration = time.time() - start_time
    print("-" * 50)
    print('Epoch {:2d} | Time {:5.4f} sec | Valid Loss {:5.4f} | Train Loss {:5.4f} '.
          format(epoch, duration, val_loss, epoch_loss / epoch_size))
    print("-" * 50)

    valid_loss_list.append(val_loss)
    train_loss_list.append(epoch_loss / epoch_size)

    if val_loss < best_valid:
        print(f"Validation loss decreased ({best_valid:.6f} --> {val_loss:.6f})....")
        print(f"Saved best model at pre_trained_models__real/{hyp_params.name}.pt \n\n")
        save_model(hyp_params, model, name=hyp_params.name)
        best_valid = val_loss
        best_epoch = epoch
        early_stopping_counter = 0  # 조기 종료 카운터 초기화
    else:
        early_stopping_counter += 1  # 검증 손실이 증가한 경우 조기 종료 카운터 증가
        print("Early Stopping Counter : ", early_stopping_counter)
        
        # 조기 종료 카운터가 지정한 횟수를 초과하면 학습 종료
        if early_stopping_counter >= 15:
            print("Validation loss has not decreased for epochs. Early stopping...")
            break

model = load_model(hyp_params, name=hyp_params.name)
test_loss_final, results, truths = evaluate(model, ctc_a2l_module, criterion, test=True)

results_round=results
results_round[:, -1] = torch.round(results_round[:, -1])
results_round[:, :-1] = (torch.round(results_round[:, :-1] * 2) / 2).float()

eval_espeak(results_round, truths)

print(f'best epoch : {best_epoch}, best valid loss: {best_valid:.5f}, test loss : {test_loss_final:.5f}')

In [ ]:
import matplotlib.pyplot as plt
# 손실 그래프 그리기

epochs = range(epoch)
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_loss_list, label='Training Loss')
plt.plot(epochs, valid_loss_list, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Seen TestSet에 대해서 모델 검증

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 모델의 파라미터 개수 출력
print(f"모델의 총 파라미터 개수: {count_parameters(model)}")


In [ ]:
model = load_model(hyp_params, name=hyp_params.name)
test_loss_final, results, truths = evaluate(model, ctc_a2l_module, criterion, test=False)

results_round=results
results_round[:, -1] = torch.round(results_round[:, -1])
results_round[:, :-1] = (torch.round(results_round[:, :-1] * 2) / 2).float()

eval_espeak(results_round, truths)

print(f'best epoch : {best_epoch}, best valid loss: {best_valid:.5f}, test loss : {test_loss_final:.5f}')